In [0]:
%run ./config

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

def create_dim_date():
    orders_path = f"{base_path}{METADATA_CONFIG['orders']['gold_path']}"
    gold_path = f"{base_path}gold/dim_date"

    df_orders = spark.read.format("delta").options(**storage_options).load(orders_path)

    min_date, max_date = (df_orders.select(
            min("OrderDate").alias("min_date"),
            max("OrderDate").alias("max_date")
        ).first()
    )

    df_date = (
        spark.sql(f"""
            SELECT explode(
                sequence(
                    to_date('{min_date}'),
                    to_date('{max_date}'),
                    interval 1 day
                )
            ) AS Date
        """)
    )

    df_date = (df_date.withColumn("DateKey",date_format(col("Date"), "yyyyMMdd").cast("int"))
        .withColumn("CalendarYear",year(col("Date")))
        .withColumn("CalendarQuarter",concat(lit("Q"), quarter(col("Date"))))
        .withColumn("MonthNumberOfYear",month(col("Date")))
        .withColumn("MonthName",date_format(col("Date"), "MMMM"))
        .withColumn("WeekNumberOfYear",weekofyear(col("Date")))
        .withColumn("DayOfMonth",dayofmonth(col("Date")))
        .withColumn("DayOfWeek",date_format(col("Date"), "EEEE"))
    )

    df_date.write.format("delta").mode("overwrite").options(**storage_options).save(gold_path)

    print("SUCCESS : dim_date created")
    print(f"Rows : {df_date.count()}")

In [0]:
from pyspark.sql.functions import *

def process_silver_to_gold(table_name):
    cfg = METADATA_CONFIG[table_name]
    silver_path = f"{base_path}{cfg['silver_path']}"
    gold_path = f"{base_path}{cfg['gold_path']}"
    try:

        df_silver = spark.read.format("delta").options(**storage_options).load(silver_path)
        if table_name == "customers":
            df_gold = (df_silver
                .select(
                    "CustomerID",
                    "FirstName",
                    "LastName",
                    "Email",
                    "Phone",
                    "City",
                    "State",
                    "start_date",
                    "end_date",
                    "is_current"
                )
            )
        elif table_name == "products":
            df_gold = (
                df_silver
                .select(
                    "ProductID",
                    "ProductName",
                    "Category",
                    "SubCategory",
                    "Brand",
                    "CostPrice"
                )
            )
        elif table_name == "orders":
            exchange_path = f"{base_path}{METADATA_CONFIG['exchange_rates']['silver_path']}"
            df_exchange = (
                spark.read
                .format("delta")
                .options(**storage_options)
                .load(exchange_path)
            )
            df_gold = (
                df_silver
                .join(
                    df_exchange,
                    (
                        (df_silver["CurrencyCode"] == df_exchange["TargetCurrency"]) &
                        (df_silver["OrderDate"] == df_exchange["RateDate"])
                    ),
                    "left"
                )
                .withColumn(
                    "LocalRevenueAmount",
                    round(col("Quantity") * col("UnitPrice"), 2)
                )
                .withColumn(
                    "USDRevenueAmount",
                    round(
                        (col("Quantity") * col("UnitPrice")) / col("ExchangeRate"),
                        2
                    )
                )
                .select(
                    "OrderID",
                    "OrderDate",
                    "CustomerID",
                    "ProductID",
                    "StoreCode",
                    "CurrencyCode",
                    "Quantity",
                    "UnitPrice",
                    "LocalRevenueAmount",
                    "ExchangeRate",
                    "USDRevenueAmount"
                )
            )

        else:
            print(f"No Gold transformation defined for {table_name}")
            return

        (
            df_gold.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .options(**storage_options)
            .save(gold_path)
        )

        print(f"SUCCESS : {table_name} loaded to Gold")
        print(f"Records Written : {df_gold.count()}")

    except Exception as e:

        print(f"FAILED : {table_name}")
        print(str(e))

In [0]:
from pyspark.sql.functions import *

def create_store_sales_summary():
    fact_path = f"{base_path}{METADATA_CONFIG['orders']['gold_path']}"
    gold_path = f"{base_path}gold/agg_store_sales"
    df_fact = (
        spark.read
        .format("delta")
        .options(**storage_options)
        .load(fact_path)
    )

    df_summary = (
        df_fact
        .groupBy("StoreCode")
        .agg(
            count("OrderID").alias("TotalOrders"),
            countDistinct("CustomerID").alias("TotalCustomers"),
            sum("Quantity").alias("TotalQuantitySold"),
            round(sum("LocalRevenueAmount"), 2).alias("TotalRevenueLocal"),
            round(sum("USDRevenueAmount"), 2).alias("TotalRevenueUSD"),
            round(avg("LocalRevenueAmount"), 2).alias("AverageOrderValue")
        )
        .orderBy(desc("TotalRevenueUSD"))
    )

    (
        df_summary.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .options(**storage_options)
        .save(gold_path)
    )

    print("SUCCESS : agg_store_sales created")
    print(f"Rows : {df_summary.count()}")

In [0]:
from pyspark.sql.functions import *

def create_customer_sales_summary():

    fact_path = f"{base_path}{METADATA_CONFIG['orders']['gold_path']}"
    gold_path = f"{base_path}gold/agg_customer_sales"

    df_fact = (
        spark.read
        .format("delta")
        .options(**storage_options)
        .load(fact_path)
    )

    df_summary = (
        df_fact
        .groupBy("CustomerID")
        .agg(
            count("OrderID").alias("TotalOrders"),
            sum("Quantity").alias("TotalQuantityPurchased"),
            round(sum("LocalRevenueAmount"), 2).alias("TotalSpendLocal"),
            round(sum("USDRevenueAmount"), 2).alias("TotalSpendUSD"),
            max("OrderDate").alias("LastOrderDate"),
            min("OrderDate").alias("FirstOrderDate"),
            round(avg("LocalRevenueAmount"), 2).alias("AverageOrderValue")
        )
        .orderBy(desc("TotalSpendUSD"))
    )

    (
        df_summary.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .options(**storage_options)
        .save(gold_path)
    )

    print("SUCCESS : agg_customer_sales created")
    print(f"Rows : {df_summary.count()}")

In [0]:
process_silver_to_gold("customers")

SUCCESS : customers loaded to Gold
Records Written : 1172


In [0]:
process_silver_to_gold("orders")

SUCCESS : orders loaded to Gold
Records Written : 843


In [0]:
process_silver_to_gold("products")

SUCCESS : products loaded to Gold
Records Written : 1404


In [0]:
create_dim_date()
create_customer_sales_summary()
create_store_sales_summary()

SUCCESS : dim_date created
Rows : 730
SUCCESS : agg_customer_sales created
Rows : 526
SUCCESS : agg_store_sales created
Rows : 50


In [0]:
spark.read.format("delta").options(**storage_options).load(f"{base_path}silver/dim_customers").display()

CustomerID,FirstName,LastName,Email,Phone,City,State,LastUpdated,_AdfPipelineRunId,_IngestionTimestamp,_ProcessedTimestamp,start_date,end_date,is_current
1846,Jeffrey,Johnson,josephdiaz@example.net,594-665-8414x63452,Samanthaview,Iowa,2025-02-23T11:27:13.000Z,Manual_Run,2026-06-27T03:05:59.882Z,2026-06-27T03:06:42.179Z,2026-06-27,null,Y
2413,Gregory,Edwards,thompsondaniel@example.net,337.658.1478x833,North Oscar,Idaho,2026-01-04T22:48:49.000Z,Manual_Run,2026-06-27T03:05:59.882Z,2026-06-27T03:06:42.179Z,2026-06-27,null,Y
1596,Andrea,Wells,mackenziefranklin@example.net,+1-453-448-5259x1630,Jimenezside,California,2025-07-19T12:46:09.000Z,Manual_Run,2026-06-27T03:05:59.882Z,2026-06-27T03:06:42.179Z,2026-06-27,null,Y
2350,Stephanie,Landry,brandon97@example.org,902-871-8272,North Robynburgh,Florida,2024-09-12T10:00:27.000Z,Manual_Run,2026-06-27T03:05:59.882Z,2026-06-27T03:06:42.179Z,2026-06-27,null,Y
1392,Brian,Jones,jacksoncraig@example.org,(976)656-8637,East Brendaburgh,Ohio,2025-09-09T00:50:37.000Z,Manual_Run,2026-06-27T03:05:59.882Z,2026-06-27T03:06:42.179Z,2026-06-27,null,Y
2155,Elizabeth,Taylor,brandontapia@example.org,(548)711-3559x495,New Jessica,Maryland,2024-07-22T01:37:06.000Z,Manual_Run,2026-06-27T03:05:59.882Z,2026-06-27T03:06:42.179Z,2026-06-27,null,Y
2343,Caroline,Erickson,ccruz@example.com,205.612.1786x435,North Robertfort,West Virginia,2025-11-23T14:06:22.000Z,Manual_Run,2026-06-27T03:05:59.882Z,2026-06-27T03:06:42.179Z,2026-06-27,null,Y
2040,Robin,Merritt,andersonjustin@example.com,+1-880-252-4392x3638,Vanceton,South Dakota,2025-10-15T05:26:13.000Z,Manual_Run,2026-06-27T03:05:59.882Z,2026-06-27T03:06:42.179Z,2026-06-27,null,Y
1631,Melissa,Sullivan,wendy81@example.com,+1-675-747-5564x065,Marquezburgh,Arkansas,2025-03-12T11:58:09.000Z,Manual_Run,2026-06-27T03:05:59.882Z,2026-06-27T03:06:42.179Z,2026-06-27,null,Y
2051,Sandra,Stevenson,sanderslynn@example.org,(638)730-8154x84329,Gordonview,Nebraska,2025-01-16T21:17:05.000Z,Manual_Run,2026-06-27T03:05:59.882Z,2026-06-27T03:06:42.179Z,2026-06-27,null,Y


In [0]:
df=spark.read.format("delta").options(**storage_options).load(f"{base_path}silver/dim_customers").select("*").groupBy("customerID").count().filter("count==1")
